# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata attribute exposes dataset-level metadata as an object
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Citation: {getattr(metadata, 'cite_as', None) if hasattr(metadata, 'cite_as') else getattr(metadata, 'citeAs', None)}")


## 2. Data Overview
Review available record sets, their fields, and related `@id`s.

In [ ]:
# List all record sets by their @id.
# Croissant exposes metadata.record_sets: a list of objects, each representing a record set schema.
record_sets = getattr(metadata, 'record_sets', None)
if record_sets is None or len(record_sets) == 0:
    # fallback for alternate property name (older mlcroissant)
    record_sets = getattr(metadata, 'recordSet', None)

print('Available Record Sets:')
record_set_ids = []
record_set_names = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', ''))
    rs_name = getattr(rs, 'name', getattr(rs, '@name', ''))
    print(f"- @id: {rs_id}, name: {rs_name}")
    record_set_ids.append(rs_id)
    record_set_names.append(rs_name)

# For each record set, list the available fields and columns (by @id)
print("\nFields and columns for each record set:")
rs_fields = dict()
for rs in record_sets:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', ''))
    fields = getattr(rs, 'fields', None) or getattr(rs, 'field', None) or []
    print(f"\nRecord Set: {rs_id}")
    field_ids = []
    for field in fields:
        f_id = getattr(field, '@id', getattr(field, 'id', ''))
        f_name = getattr(field, 'name', getattr(field, '@name', ''))
        print(f"  Field @id: {f_id}, name: {f_name}")
        field_ids.append(f_id)
        # For each field, check for columns
        columns = getattr(field, 'columns', None) or getattr(field, 'column', None) or []
        for col in columns:
            c_id = getattr(col, '@id', getattr(col, 'id', ''))
            c_name = getattr(col, 'name', getattr(col, '@name', ''))
            print(f"    Column @id: {c_id}, name: {c_name}")
    rs_fields[rs_id] = field_ids


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data for all available record sets by @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        print(f"Loading data for record set: {record_set_id}")
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if len(records) == 0:
            print(f"  No records found for record set {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded DataFrame with shape {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        # Print first few records (optional)
        display(df.head())
    except Exception as e:
        print(f"  Error for record set {record_set_id}: {e}")
        continue
# If only one record set is present, select it as default for downstream exploration
if len(dataframes) > 0:
    default_record_set_id = list(dataframes.keys())[0]
    print(f"Default record_set_id for analysis: {default_record_set_id}")
else:
    default_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
import numpy as np

# Choose a numeric field to analyze
if default_record_set_id is not None:
    df = dataframes[default_record_set_id]
    print(f"Analyzing record set: {default_record_set_id}\nAvailable columns: {df.columns.tolist()}")
    # Try to automatically select a numeric field (e.g., column containing 'MW' or 'abundance')
    candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if len(candidates) == 0:
        # Try to cast numeric-looking columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                continue
        candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    
    if len(candidates) > 0:
        numeric_field = candidates[0]
        print(f"Selected numeric field: {numeric_field}")
        # Set threshold as the 75th percentile for demonstration
        threshold = df[numeric_field].dropna().quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by another categorical field, if present
        group_candidates = [col for col in df.columns if df[col].nunique() < min(20, len(df)//4) and col != numeric_field]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data:")
            display(grouped_df.head())
        else:
            print("No suitable categorical/group field found for grouping.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No data found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if default_record_set_id is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If group_field is available
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization not available (numeric field or record set unavailable).")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.